# Data Cleaning

This notebook cleans the raw Vietnam history CSV files and exports dashboard-ready files to `data/processed/`. It preserves the original raw files and creates normalized relationship tables for multi-value event and location references.

In [ ]:
from pathlib import Path
import json
import pandas as pd

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data' / 'raw' / 'events.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing data/raw/events.csv')

ROOT = find_project_root()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

files = {
    'events': RAW_DIR / 'events.csv',
    'people': RAW_DIR / 'people.csv',
    'periods': RAW_DIR / 'periods.csv',
    'locations': RAW_DIR / 'locations.csv',
}
data = {name: pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig') for name, path in files.items()}
{name: df.shape for name, df in data.items()}

In [ ]:
def clean_text_frame(df):
    cleaned = df.copy()
    for col in cleaned.columns:
        cleaned[col] = cleaned[col].astype(str).str.strip()
        cleaned[col] = cleaned[col].str.replace(r'\s+', ' ', regex=True)
    return cleaned

def to_int_nullable(series):
    return pd.to_numeric(series.replace('', pd.NA), errors='coerce').astype('Int64')

def to_float_nullable(series):
    return pd.to_numeric(series.replace('', pd.NA), errors='coerce')

def split_ids(value):
    value = str(value).strip()
    if not value:
        return []
    return [part.strip() for part in value.replace('|', ';').split(';') if part.strip()]

def split_names(value):
    value = str(value).strip()
    if not value:
        return []
    return [part.strip() for part in value.replace('|', ';').split(';') if part.strip()]

In [ ]:
events = clean_text_frame(data['events'])
people = clean_text_frame(data['people'])
periods = clean_text_frame(data['periods'])
locations = clean_text_frame(data['locations'])

for col in ['year_start', 'year_end', 'importance_score']:
    events[col] = to_int_nullable(events[col])

for col in ['first_mentioned_year', 'last_mentioned_year', 'birth_year', 'death_year', 'age_at_death_estimate', 'importance_score']:
    people[col] = to_int_nullable(people[col])

for col in ['start_year', 'end_year']:
    periods[col] = to_int_nullable(periods[col])

locations['latitude'] = to_float_nullable(locations['latitude'])
locations['longitude'] = to_float_nullable(locations['longitude'])
locations['first_mentioned_year'] = to_int_nullable(locations['first_mentioned_year'])

In [ ]:
# Stable sort order keeps dashboards and diffs predictable.
events = events.sort_values(['year_start', 'event_id'], na_position='last').reset_index(drop=True)
periods = periods.sort_values(['start_year', 'period_id'], na_position='last').reset_index(drop=True)
people = people.sort_values(['first_mentioned_year', 'person_id'], na_position='last').reset_index(drop=True)
locations = locations.sort_values(['first_mentioned_year', 'location_id'], na_position='last').reset_index(drop=True)

In [ ]:
# Normalize multi-value references into relationship tables.
event_people_rows = []
for _, row in events.iterrows():
    ids = split_ids(row['person_ids'])
    names = split_names(row['person_names'])
    for index, person_id in enumerate(ids):
        event_people_rows.append({
            'event_id': row['event_id'],
            'person_id': person_id,
            'person_name': names[index] if index < len(names) else '',
        })
event_people = pd.DataFrame(event_people_rows, columns=['event_id', 'person_id', 'person_name'])

event_location_rows = []
for _, row in events.iterrows():
    ids = split_ids(row['location_ids'])
    names = split_names(row['location_names'])
    for index, location_id in enumerate(ids):
        event_location_rows.append({
            'event_id': row['event_id'],
            'location_id': location_id,
            'location_name': names[index] if index < len(names) else '',
        })
event_locations = pd.DataFrame(event_location_rows, columns=['event_id', 'location_id', 'location_name'])

location_period_rows = []
for _, row in locations.iterrows():
    for period_id in split_ids(row['related_period_ids']):
        location_period_rows.append({
            'location_id': row['location_id'],
            'period_id': period_id,
        })
location_periods = pd.DataFrame(location_period_rows, columns=['location_id', 'period_id'])

event_people.shape, event_locations.shape, location_periods.shape

In [ ]:
outputs = {
    'events_clean.csv': events,
    'people_clean.csv': people,
    'periods_clean.csv': periods,
    'locations_clean.csv': locations,
    'event_people.csv': event_people,
    'event_locations.csv': event_locations,
    'location_periods.csv': location_periods,
}

for filename, df in outputs.items():
    df.to_csv(PROCESSED_DIR / filename, index=False, encoding='utf-8-sig', na_rep='')

cleaning_report = {
    'raw_rows': {name: int(len(df)) for name, df in data.items()},
    'processed_rows': {filename: int(len(df)) for filename, df in outputs.items()},
    'output_files': list(outputs.keys()),
}
(PROCESSED_DIR / 'cleaning_report.json').write_text(json.dumps(cleaning_report, indent=2, ensure_ascii=False), encoding='utf-8')

cleaning_report